# Lab 23-04 · DRAM I/O: Termination, Reflections, Equalization and PAM

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
from itertools import product
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import ddrio as dd


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Termination and reflections in ngspice

*Predict* each value first.

In [2]:
vl, i, p = dd.pod_low(1.1, 34, 40)
arr120 = dd.far_end_arrivals(1.0, 34, 40, 120, 3)
arropen = dd.far_end_arrivals(1.0, 34, 40, math.inf, 3)
h = dd.rc_cursors(1 / 8.8e9, 80e-12, 3)
pred = {"vpod": vl, "ipod_ma": i * 1e3, "ppod_mw": p * 1e3, "vlv": dd.lvstl_high(0.5, 40, 40)[0],
        "f1_first": dd.launched_wave(1.0, 34, 40), "f2_first": arr120[0], "f2_second": arr120[1],
        "f3_first": arropen[0], "f3_second": arropen[1], "h0": h[0], "h1": h[1], "h2": h[2]}
sim = ngspice("dram_io.cir")
for kk, vv in pred.items():
    print(f"{kk:9s}: model {vv:.5f}, ngspice {sim[kk]:.5f}")

vpod     : model 0.50541, ngspice 0.50541
ipod_ma  : model 14.86486, ngspice 14.86486
ppod_mw  : model 16.35135, ngspice 16.35135
vlv      : model 0.25000, ngspice 0.25000
f1_first : model 0.54054, ngspice 0.54054
f2_first : model 0.81081, ngspice 0.81081
f2_second: model 0.77794, ngspice 0.77794
f3_first : model 1.08108, ngspice 1.08108
f3_second: model 0.99343, ngspice 0.99343
h0       : model 0.75840, ngspice 0.75838
h1       : model 0.18323, ngspice 0.18324
h2       : model 0.04427, ngspice 0.04427


## Part 2 · Data bus inversion

*Predict* the largest number of low wires per byte with and without DBI.

In [3]:
def lows(bits, dbi):
    z = bits.count(0)
    if dbi and z > 4:
        return (8 - z) + 1
    return z


all_bytes = list(product((0, 1), repeat=8))
for dbi in (False, True):
    counts = [lows(b, dbi) for b in all_bytes]
    print(f"DBI {dbi!s:5s}: average {np.mean(counts):.4f}, maximum {max(counts)}")
q = 0.7
for dbi in (False, True):
    e = sum(q ** b.count(0) * (1 - q) ** b.count(1) * lows(b, dbi) for b in all_bytes)
    print(f"biased data (P(0) = {q}), DBI {dbi!s:5s}: {e:.4f} low wires per byte")

DBI False: average 4.0000, maximum 8
DBI True : average 3.2695, maximum 4
biased data (P(0) = 0.7), DBI False: 5.6000 low wires per byte
biased data (P(0) = 0.7), DBI True : 3.0647 low wires per byte


## Part 3 · NRZ, PAM3 and PAM4 through a first-order channel

*Predict* which scheme has the tallest eye at τ = 15 ps.

In [4]:
BIT = 48e9
schemes = {"NRZ": (2, 1.0), "PAM3": (3, 1.5), "PAM4": (4, 2.0)}


def closed_form(tau, m, b):
    a = math.exp(-(b / BIT) / tau)
    return dd.worst_eye(1 - a, a, m)


def simulated_eye(tau, m, b, n=200_000, taps=12, seed=2304):
    rng = np.random.default_rng(seed)
    levels = np.linspace(-1, 1, m)
    sym = rng.integers(0, m, n)
    h = np.array(dd.rc_cursors(b / BIT, tau, taps))
    rx = np.convolve(levels[sym], h)[: n]
    gaps = []
    for k in range(m - 1):  # the eye between levels k and k+1
        lo = rx[taps:][sym[taps:] == k].max()
        hi = rx[taps:][sym[taps:] == k + 1].min()
        gaps.append((hi - lo) / 2)
    return min(gaps)


for name, (m, b) in schemes.items():
    print(f"{name:4s}: closed form {closed_form(15e-12, m, b):.4f}, random sequence {simulated_eye(15e-12, m, b):.4f}")
taus = np.linspace(5e-12, 32e-12, 60)
plt.figure()
for name, (m, b) in schemes.items():
    plt.plot(taus * 1e12, [closed_form(t, m, b) for t in taus], label=name)
plt.axhline(0, color="0.5", lw=0.8); plt.xlabel("channel time constant (ps)"); plt.ylabel("worst-case eye (fraction of swing)")
plt.legend(); plt.show()

NRZ : closed form 0.5013, random sequence 0.5013
PAM3: closed form 0.3132, random sequence 0.3132
PAM4: closed form 0.2504, random sequence 0.2504


## Part 4 · The families' bandwidths

*Predict* how many pins each option needs for 1 TB/s.

In [5]:
rows = [("DDR5-6400 DIMM channel", 6400e6, 64), ("LPDDR5X-9600 x16 channel", 9600e6, 16),
        ("GDDR7 32 Gb/s x32 device", 32e9, 32), ("RTX 5090 bus (28 Gb/s, 512-bit)", 28e9, 512)]
for name, r, w in rows:
    bw = dd.bandwidth(r, w)
    print(f"{name:32s}: {bw/1e9:8.1f} GB/s; for 1 TB/s: {math.ceil(1e12/bw):3d} units, {math.ceil(1e12/bw)*w:5d} pins")

DDR5-6400 DIMM channel          :     51.2 GB/s; for 1 TB/s:  20 units,  1280 pins
LPDDR5X-9600 x16 channel        :     19.2 GB/s; for 1 TB/s:  53 units,   848 pins
GDDR7 32 Gb/s x32 device        :    128.0 GB/s; for 1 TB/s:   8 units,   256 pins
RTX 5090 bus (28 Gb/s, 512-bit) :   1792.0 GB/s; for 1 TB/s:   1 units,   512 pins
